In [19]:
import importlib
import functional_functions as ff
from cloudpathlib import S3Path, S3Client
importlib.reload(ff)
from nilearn.maskers import SurfaceLabelsMasker
from nilearn.connectome import ConnectivityMeasure
from nilearn.surface import SurfaceImage
import numpy as np
import pandas as pd
from pathlib import Path

In [3]:
# create mesh and parcellate based on MMP
labels_image = ff.create_mmp_mesh()

In [4]:
# create look up table with labels 
lut = ff.create_mmp_lookup()

In [5]:
# make and fit the masker to the ROIs from Glasser 
surface_lbls_masker = SurfaceLabelsMasker(labels_img=labels_image, lut=lut).fit()

In [6]:
# read in participant csv 
df = pd.read_csv('train_split.csv')
subjects = df['Subject'].values

In [7]:
# instantiate storage matrix 
mat = []

In [20]:
# create HCP base path
cache_path = Path('/tmp/cache') 
if not cache_path.exists(): 
    cache_path.mkdir()
hcp_base_path = S3Path('s3://hcp-openaccess/HCP_1200/', 
                       client = S3Client(local_cache_dir = cache_path, profile_name = 'hcp'))
hcp_base_path

S3Path('s3://hcp-openaccess/HCP_1200/')

In [22]:
# instantiate matrices for connectivity data split by block type
win_mat = [] 
loss_mat = [] 
TR_LEN = 0.72

# create HCP base path
cache_path = Path('/tmp/cache') 
if not cache_path.exists(): 
    cache_path.mkdir()

correlation_measure = ConnectivityMeasure(kind='correlation', vectorize=True)

# loop through all participants 
for sub_num in subjects:
    
    (all_left_hemi, all_right_hemi) = ff.ciis_to_giis(sub_num)
    
    # put our data in SurfaceImage object 
    func_data = SurfaceImage(
        mesh=labels_image.mesh,
        data= {
            "left": all_left_hemi.T,
            "right": all_right_hemi.T
        }
    )
    roi_time_series = surface_lbls_masker.transform(func_data)

    # split roi_time_series into separate dfs for win/loss blocks
    win_time_series, loss_time_series = ff.split_time_series(
        roi_time_series=roi_time_series,
        patient_num=sub_num,
        hcp_base_path=hcp_base_path,
        tr_len=TR_LEN,
    )
    
    # make correlation matrix 
    correlation_matrix_win = correlation_measure.fit_transform([win_time_series])[0]
    correlation_matrix_loss = correlation_measure.fit_transform([loss_time_series])[0]


    # add subject number to beginning of vectorized corrmat
    # add to matrix
    win_mat.append(np.insert(correlation_matrix_win, 0, sub_num))
    loss_mat.append(np.insert(correlation_matrix_loss, 0, sub_num))

    print(sub_num)

# save to home directory
home = Path.home()
np.save(home/'win_training_corrmats_all.npy', win_mat)
np.save(home/'loss_training_corrmats_all.npy', loss_mat)

pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
/tmp/ipykernel_135/2819674158.py:37: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore' option will be removed. and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  correlation_matrix_win = correlation_measure.fit_transform([win_time_series])[0]
/tmp/ipykernel_135/2819674158.py:38: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced

163129


pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
/tmp/ipykernel_135/2819674158.py:37: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore' option will be removed. and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  correlation_matrix_win = correlation_measure.fit_transform([win_time_series])[0]
/tmp/ipykernel_135/2819674158.py:38: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced

308129


pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
/tmp/ipykernel_135/2819674158.py:37: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore' option will be removed. and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  correlation_matrix_win = correlation_measure.fit_transform([win_time_series])[0]
/tmp/ipykernel_135/2819674158.py:38: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced

...

159441


pixdim[1,2,3] should be non-zero; setting 0 dims to 1
pixdim[1,2,3] should be non-zero; setting 0 dims to 1
/tmp/ipykernel_135/2819674158.py:37: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore' option will be removed. and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  correlation_matrix_win = correlation_measure.fit_transform([win_time_series])[0]
/tmp/ipykernel_135/2819674158.py:38: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced

616645


pixdim[1,2,3] should be non-zero; setting 0 dims to 1


KeyboardInterrupt: 

In [45]:
np.shape(win_mat)
np.shape(loss_mat)

(692, 64981)

In [46]:
# save to home directory ----> moved up to save output from run 
#home = Path.home()
#np.save(home/'win_training_corrmats_all.npy', win_mat)
$np.save(home/'loss_training_corrmats_all.npy', loss_mat)